# Avaliacao de Qualidade do Retrieval - Antes vs. Depois do Re-ranking

Este notebook compara os resultados da busca **antes** e **depois** da aplicacao
do re-ranking, para provar com exemplos que o re-ranking traz os chunks mais
relevantes para o topo.

Seguimos o Entregavel #3 do projeto: comparar a busca em pelo menos **3 consultas
de teste** diferentes.

> **ANTES**: ordem crua por similaridade semantica (so o embedding).
>
> **DEPOIS**: ordem apos o re-ranking hibrido (60% semantico + 40% lexical).

## 0. Setup

Importamos as funcoes **reais** do codigo de producao (`src/rag/retrieval.py`):

- `retrieve_chunks`: busca os chunks mais similares (SEM re-ranking)
- `rerank_chunks`: aplica o re-ranking hibrido sobre os chunks recuperados

> **Pre-requisito:** o ChromaDB precisa estar populado com TODOS os chunks.
> Rode antes, na raiz do projeto: `python -m src.rag.ingestion`.
> O notebook didatico de ingestion insere so 5 chunks de exemplo (e recria a
> collection), entao nao use ele para popular o banco real.

Como este notebook esta em `notebooks/`, subimos um nivel para a raiz do projeto
antes de importar - assim os caminhos relativos (ChromaDB) funcionam.

In [1]:
import os
import sys
from pathlib import Path

# O notebook esta em notebooks/ -> sobe um nivel para a raiz do projeto.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

# Garante que o pacote 'src' seja importavel
sys.path.insert(0, str(Path.cwd()))

from src.rag.retrieval import (
    retrieve_chunks,
    rerank_chunks,
    TOP_K_RETRIEVAL,
    TOP_K_FINAL,
)

print(f"Working directory: {Path.cwd()}")
print(f"Recupera {TOP_K_RETRIEVAL} candidatos, re-ranking devolve top {TOP_K_FINAL}.")
print("Setup concluido.")

Working directory: c:\Users\BB442HD\OneDrive - EY\Desktop\compliance-viewer
Recupera 10 candidatos, re-ranking devolve top 3.
Setup concluido.


## 1. Montando a comparacao

Para cada query fazemos o seguinte:

1. Recuperamos os `TOP_K_RETRIEVAL` (10) candidatos **uma unica vez**.
2. **ANTES**: pegamos os 3 primeiros na ordem crua de similaridade.
3. **DEPOIS**: aplicamos o re-ranking hibrido e pegamos os 3 melhores.
4. Mostramos as duas listas lado a lado e indicamos se a ordem mudou.

Recuperar os candidatos uma vez so garante que as duas listas partem
exatamente do mesmo conjunto - a unica diferenca entre elas e o re-ranking.

In [2]:
def comparar(query, top_n=TOP_K_FINAL):
    # Recupera os candidatos (top 10) uma unica vez
    candidatos = retrieve_chunks(query, top_k=TOP_K_RETRIEVAL)

    # ANTES: ordem crua por similaridade semantica (como o Chroma devolveu)
    antes = candidatos[:top_n]

    # DEPOIS: aplica o re-ranking hibrido e pega os melhores
    depois = rerank_chunks(query, candidatos)[:top_n]

    print("=" * 72)
    print(f"QUERY: {query}")
    print("=" * 72)

    print(f"--- ANTES do re-ranking (top {top_n} por similaridade) ---")
    for i, c in enumerate(antes, 1):
        print(f"{i}. [{c['source']} #{c['chunk_index']}]  sim={c['similarity_score']:.3f}")
        print(f"     {c['text'][:110]}...")

    print(f"--- DEPOIS do re-ranking (top {top_n} por score_final) ---")
    for i, c in enumerate(depois, 1):
        print(f"{i}. [{c['source']} #{c['chunk_index']}]  sim={c['similarity_score']:.3f}  final={c['score_final']:.3f}")
        print(f"     {c['text'][:110]}...")

    id_antes = [(c["source"], c["chunk_index"]) for c in antes]
    id_depois = [(c["source"], c["chunk_index"]) for c in depois]
    if id_antes == id_depois:
        print(">> O re-ranking NAO mudou a ordem para esta query.")
    else:
        print(">> O re-ranking MUDOU a ordem dos resultados.")
    print()

## 2. As tres consultas de teste

Escolhemos 3 perguntas que cobrem os perfis de risco e a parte regulatoria.
Perguntas com termos especificos (ex: "conservador", "suitability") sao as que
mais se beneficiam do componente lexical do re-ranking.

In [3]:
queries_teste = [
    "Quais sao as regras para um perfil conservador?",
    "Quais produtos sao adequados para um investidor arrojado?",
    "O que diz a regulamentacao sobre suitability e adequacao de perfil?",
]

for q in queries_teste:
    comparar(q)

QUERY: Quais sao as regras para um perfil conservador?
--- ANTES do re-ranking (top 3 por similaridade) ---
1. [politica_adequacao_investimento_v1.2.txt #1]  sim=0.675
     **2. Classificação de Perfis de Cliente**
- **2.1. Conservador:** Clientes com baixa tolerância a risco, prior...
2. [email_analise_cliente_01.txt #2]  sim=0.615
     O que acha de agendarmos uma conversa?"
---

**Checklist de Análise Rápida:**
- [ ] A recomendação está alinha...
3. [anbima_codigo_distribuicao_produtos_Investimento.pdf #131]  sim=0.611
     Art. 53. As instituições participantes devem implementar e manter, em documento escrito, 
regras, procedimento...
--- DEPOIS do re-ranking (top 3 por score_final) ---
1. [anbima_codigo_distribuicao_produtos_Investimento.pdf #131]  sim=0.611  final=0.607
     Art. 53. As instituições participantes devem implementar e manter, em documento escrito, 
regras, procedimento...
2. [email_analise_cliente_01.txt #2]  sim=0.615  final=0.529
     O que acha de agendarmos uma

## 3. Conclusao

Em quais queries o re-ranking mudou a ordem? Na query "Quais são as regras para um perfil conservador?", o re-ranking alterou o top 3. A busca por similaridade pura colocava no topo um texto genérico de classificação de perfil e um e-mail de cliente (ruído). Após o re-ranking, a ordem mudou.

Os chunks que subiram são mais relevantes? Sim. O artigo normativo da ANBIMA (Art. 53 — "as instituições devem implementar e manter, em documento escrito, regras, procedimentos...") estava em 3º lugar na similaridade pura (sim=0.611) e subiu para 1º após o re-ranking (final=0.607). É a fonte regulatória mais adequada à pergunta — exatamente o que deve embasar uma resposta de compliance.

O componente lexical (40%) ajudou? Sim, e foi o fator decisivo aqui. A pergunta contém o termo "regras", que aparece literalmente no chunk da ANBIMA. A similaridade semântica pura tinha rebaixado esse chunk; o score lexical reconheceu o termo técnico e o promoveu ao topo. Confirma o papel do lexical: capturar terminologia normativa que a semântica sozinha perde.